# Outbreak Detection Analysis

This notebook runs the pipeline and visualizes generated alerts and model feature importances.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parent
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from outbreak_detection.pipeline import run_pipeline

sns.set_theme(style='whitegrid')

## Run Pipeline

In [ ]:
summary = run_pipeline(str(ROOT / 'config.yaml'))
summary

## Load Outputs

In [ ]:
alerts = pd.read_csv(ROOT / 'outputs' / 'alerts.csv', parse_dates=['report_date'])
features = pd.read_csv(ROOT / 'data' / 'processed' / 'features.csv', parse_dates=['report_date'])
alerts.head()

## Daily Alert Counts

In [ ]:
daily = alerts[alerts['is_alert'] == 1].groupby(['report_date', 'region'], as_index=False)['is_alert'].sum()

plt.figure(figsize=(12, 5))
sns.lineplot(data=daily, x='report_date', y='is_alert', hue='region')
plt.title('Daily Alert Count by Region')
plt.xlabel('Date')
plt.ylabel('Alert Count')
plt.tight_layout()
plt.show()

## Top Feature Importances

In [ ]:
model = joblib.load(ROOT / 'models' / 'outbreak_model.joblib')
feature_names = [c for c in features.columns if c.endswith(tuple(['_lag_1', '_lag_3', '_lag_7', '_rollmean_3', '_rollmean_7', '_rollmean_14', '_rollstd_3', '_rollstd_7', '_rollstd_14']))]
importances = pd.Series(model.feature_importances_, index=feature_names).sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(x=importances.values, y=importances.index, orient='h')
plt.title('Top 15 Feature Importances')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()